In [68]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report
from collections import defaultdict, Counter
import unicodedata
import os

fifa_ranking_2022 = pd.read_csv('fifa_ranking_2022-10-06.csv')
fifa_ranking_2026 = pd.read_csv('fifa_ranking_2026-06-08.csv')
matches           = pd.read_csv('matches_1930_2022.csv')
schedule_2026     = pd.read_csv('schedule_2026.csv')
world_cup         = pd.read_csv('world_cup.csv')

print('Successfully loaded all datasets.')

Successfully loaded all datasets.


In [69]:
matches['Date'] = pd.to_datetime(matches['Date'])
matches = matches.sort_values('Date').reset_index(drop=True)
print(matches[['Date', 'home_team', 'away_team']].head())

        Date      home_team away_team
0 1930-07-13         France    Mexico
1 1930-07-13  United States   Belgium
2 1930-07-14     Yugoslavia    Brazil
3 1930-07-14        Romania      Peru
4 1930-07-15      Argentina    France


In [70]:
team_mapping = {
    'Czechia': 'Czech Republic',
    'United States': 'USA',
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',
    'Congo DR': 'Congo',
    'Curaçao': 'Curacao',
}

def clean_team(x):
    return str(x).strip()

for df in [matches, schedule_2026]:
    df['home_team'] = df['home_team'].replace(team_mapping).apply(clean_team)
    df['away_team'] = df['away_team'].replace(team_mapping).apply(clean_team)

print('Team names cleaned.')

Team names cleaned.


In [71]:
INITIAL_ELO    = 1500
REFERENCE_DATE = pd.Timestamp('2026-06-01')
HALF_LIFE_DAYS = 365 * 4   # influence halves every 4 years
K_BASE         = 30

elo = defaultdict(lambda: INITIAL_ELO)

matches['home_elo'] = 0.0
matches['away_elo'] = 0.0
matches['elo_diff'] = 0.0

def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def mov_multiplier(goal_diff, elo_diff_before):
    if goal_diff == 0:
        return 1.0
    raw        = np.log(goal_diff + 1) + 1
    correction = 2.2 / (elo_diff_before * 0.001 + 2.2) if elo_diff_before > 0 else 1.0
    return raw * correction

def k_factor(match_date):
    days_ago = (REFERENCE_DATE - match_date).days
    decay    = 0.5 ** (days_ago / HALF_LIFE_DAYS)
    return K_BASE * decay

for idx, row in matches.iterrows():
    home, away          = row['home_team'], row['away_team']
    home_rating         = elo[home]
    away_rating         = elo[away]

    matches.loc[idx, 'home_elo'] = home_rating
    matches.loc[idx, 'away_elo'] = away_rating
    matches.loc[idx, 'elo_diff'] = home_rating - away_rating

    exp_home = expected_score(home_rating, away_rating)
    exp_away = expected_score(away_rating, home_rating)

    hg, ag = row['home_score'], row['away_score']
    if hg > ag:
        actual_home, actual_away = 1, 0
        winner_elo_diff = home_rating - away_rating
    elif hg < ag:
        actual_home, actual_away = 0, 1
        winner_elo_diff = away_rating - home_rating
    else:
        actual_home, actual_away = 0.5, 0.5
        winner_elo_diff = 0

    K   = k_factor(row['Date'])
    mov = mov_multiplier(abs(hg - ag), winner_elo_diff)

    elo[home] = home_rating + K * mov * (actual_home - exp_home)
    elo[away] = away_rating + K * mov * (actual_away - exp_away)

elo_table = pd.DataFrame({'team': list(elo.keys()), 'elo': list(elo.values())})
print(elo_table.sort_values('elo', ascending=False).head(20).to_string(index=False))

       team         elo
     France 1598.679005
Netherlands 1570.410380
  Argentina 1563.021788
     Brazil 1554.871962
    England 1543.391399
    Belgium 1536.253479
    Croatia 1530.001905
    Germany 1526.311961
   Portugal 1524.056345
   Colombia 1520.306274
      Spain 1520.071994
    Uruguay 1518.566163
     Sweden 1507.867978
    Morocco 1506.551557
     Russia 1505.671316
      Japan 1503.467020
      Chile 1503.306253
    Ecuador 1503.019396
    Türkiye 1500.968319
      Italy 1500.943426


In [72]:
matches['home_xg'] = pd.to_numeric(matches['home_xg'], errors='coerce')
matches['away_xg'] = pd.to_numeric(matches['away_xg'], errors='coerce')
matches['home_red_card'] = pd.to_numeric(matches['home_red_card'], errors='coerce').fillna(0)
matches['away_red_card'] = pd.to_numeric(matches['away_red_card'], errors='coerce').fillna(0)

# Use xG where available, fall back to actual goals
matches['home_xg_filled'] = matches['home_xg'].fillna(matches['home_score'])
matches['away_xg_filled'] = matches['away_xg'].fillna(matches['away_score'])

# Stage flag
knockout_rounds = {
    'Round of 16', 'Quarter-finals', 'Semi-finals',
    'Final', 'Third-place match', 'Second round'
}
matches['is_knockout'] = matches['Round'].isin(knockout_rounds).astype(int)

print('xG available for', matches['home_xg'].notna().sum(), 'matches')
print('Knockout matches:', matches['is_knockout'].sum())

xG available for 128 matches
Knockout matches: 270


In [73]:
def calculate_form_weighted(history, window=10):
    """Exponentially-weighted form over last `window` games (decay=0.9)."""
    if len(history) == 0:
        return {
            'win_rate': 0, 'draw_rate': 0, 'loss_rate': 0,
            'goals_for': 0, 'goals_against': 0,
            'xg_for': 0, 'xg_against': 0, 'red_cards': 0
        }
    recent  = history[-window:]
    n       = len(recent)
    weights = np.array([0.9 ** (n - 1 - i) for i in range(n)])
    weights /= weights.sum()

    def wavg(key):
        return float(np.dot(weights, [h[key] for h in recent]))

    return {
        'win_rate':      wavg('win'),
        'draw_rate':     wavg('draw'),
        'loss_rate':     wavg('loss'),
        'goals_for':     wavg('goals_for'),
        'goals_against': wavg('goals_against'),
        'xg_for':        wavg('xg_for'),
        'xg_against':    wavg('xg_against'),
        'red_cards':     wavg('red_cards'),
    }

# Alias for compatibility
calculate_form = calculate_form_weighted
print('Weighted form function defined.')

Weighted form function defined.


In [74]:
new_cols = [
    'home_last5_win_rate', 'away_last5_win_rate',
    'home_last5_draw_rate', 'away_last5_draw_rate',
    'home_last5_loss_rate', 'away_last5_loss_rate',
    'home_last5_goals_for', 'away_last5_goals_for',
    'home_last5_goals_against', 'away_last5_goals_against',
    'home_xg_for', 'away_xg_for',
    'home_xg_against', 'away_xg_against',
    'home_red_cards_avg', 'away_red_cards_avg',
]
for c in new_cols:
    matches[c] = 0.0

team_history = defaultdict(list)

for idx, row in matches.iterrows():
    home, away = row['home_team'], row['away_team']
    hf = calculate_form_weighted(team_history[home])
    af = calculate_form_weighted(team_history[away])

    matches.loc[idx, 'home_last5_win_rate']      = hf['win_rate']
    matches.loc[idx, 'away_last5_win_rate']       = af['win_rate']
    matches.loc[idx, 'home_last5_draw_rate']      = hf['draw_rate']
    matches.loc[idx, 'away_last5_draw_rate']      = af['draw_rate']
    matches.loc[idx, 'home_last5_loss_rate']      = hf['loss_rate']
    matches.loc[idx, 'away_last5_loss_rate']      = af['loss_rate']
    matches.loc[idx, 'home_last5_goals_for']      = hf['goals_for']
    matches.loc[idx, 'away_last5_goals_for']      = af['goals_for']
    matches.loc[idx, 'home_last5_goals_against']  = hf['goals_against']
    matches.loc[idx, 'away_last5_goals_against']  = af['goals_against']
    matches.loc[idx, 'home_xg_for']               = hf['xg_for']
    matches.loc[idx, 'away_xg_for']               = af['xg_for']
    matches.loc[idx, 'home_xg_against']           = hf['xg_against']
    matches.loc[idx, 'away_xg_against']           = af['xg_against']
    matches.loc[idx, 'home_red_cards_avg']        = hf['red_cards']
    matches.loc[idx, 'away_red_cards_avg']        = af['red_cards']

    home_win  = int(row['home_score'] > row['away_score'])
    home_draw = int(row['home_score'] == row['away_score'])

    team_history[home].append({
        'win': home_win, 'draw': home_draw, 'loss': 1 - home_win - home_draw,
        'goals_for': row['home_score'], 'goals_against': row['away_score'],
        'xg_for': row['home_xg_filled'], 'xg_against': row['away_xg_filled'],
        'red_cards': row['home_red_card'],
    })
    team_history[away].append({
        'win': 1 - home_win - home_draw, 'draw': home_draw, 'loss': home_win,
        'goals_for': row['away_score'], 'goals_against': row['home_score'],
        'xg_for': row['away_xg_filled'], 'xg_against': row['home_xg_filled'],
        'red_cards': row['away_red_card'],
    })

matches['form_diff']      = matches['home_last5_win_rate']      - matches['away_last5_win_rate']
matches['attack_diff']    = matches['home_last5_goals_for']     - matches['away_last5_goals_for']
matches['defense_diff']   = matches['away_last5_goals_against'] - matches['home_last5_goals_against']
matches['draw_rate_diff'] = matches['home_last5_draw_rate']     - matches['away_last5_draw_rate']
matches['xg_diff']        = matches['home_xg_for']              - matches['away_xg_for']

print('Team history built for', len(team_history), 'teams.')

Team history built for 86 teams.


In [75]:
# Map our internal names → FIFA file names
fifa_name_map = {
    'Korea Republic': 'South Korea',
    'IR Iran':        'Iran',
    'Türkiye':        'Turkey',
    "Côte d'Ivoire": 'Ivory Coast',
    'USA':            'United States',
    'Congo':          'Congo DR',
}

fifa26 = fifa_ranking_2026.copy()
inv_map = {v: k for k, v in fifa_name_map.items()}
fifa26['team'] = fifa26['team'].replace(inv_map)

max_rank = fifa26['rank'].max()
fifa26['rank_score'] = 1 - (fifa26['rank'] - 1) / max_rank

fifa_rank_dict  = dict(zip(fifa26['team'], fifa26['rank']))
fifa_pts_dict   = dict(zip(fifa26['team'], fifa26['points']))
fifa_score_dict = dict(zip(fifa26['team'], fifa26['rank_score']))
fifa_conf_dict  = dict(zip(fifa26['team'], fifa26['association']))

# Build enriched team_dict
team_features_list = []
for team, history in team_history.items():
    stats = calculate_form_weighted(history)
    team_features_list.append({
        'team':          team,
        'elo':           elo[team],
        'win_rate':      stats['win_rate'],
        'draw_rate':     stats['draw_rate'],
        'loss_rate':     stats['loss_rate'],
        'goals_for':     stats['goals_for'],
        'goals_against': stats['goals_against'],
        'xg_for':        stats['xg_for'],
        'xg_against':    stats['xg_against'],
        'red_cards':     stats['red_cards'],
        'fifa_rank':     fifa_rank_dict.get(team, 150),
        'fifa_pts':      fifa_pts_dict.get(team, 1000),
        'fifa_score':    fifa_score_dict.get(team, 0.0),
    })

team_features = pd.DataFrame(team_features_list)
team_dict = {row['team']: row.to_dict() for _, row in team_features.iterrows()}

# Coverage check
all_2026_teams = set(schedule_2026['home_team']).union(set(schedule_2026['away_team']))
missing = sorted([t for t in all_2026_teams if t not in fifa_rank_dict])
print('Teams missing FIFA rank (defaulting to 150):', missing)
print('Top 10 by FIFA rank:')
print(fifa26.sort_values('rank').head(10)[['team','rank','points']].to_string(index=False))

Teams missing FIFA rank (defaulting to 150): ['Cape Verde', 'Curacao', 'Czech Republic']
Top 10 by FIFA rank:
       team  rank      points
  Argentina     1 1876.118331
      Spain     2 1873.013187
     France     3 1869.428449
    England     4 1827.048678
   Portugal     5 1766.177547
     Brazil     6 1765.856297
    Morocco     7 1755.100232
Netherlands     8 1751.097835
    Belgium     9 1742.235945
    Germany    10 1735.771984


In [76]:
conf_strength = {'UEFA': 5, 'CONMEBOL': 4, 'CONCACAF': 3, 'CAF': 2, 'AFC': 2, 'OFC': 1}

def conf_score(team):
    conf = fifa_conf_dict.get(team, 'AFC')
    return conf_strength.get(conf, 2)

# Rest days
last_match_date = {}
matches['home_rest_days'] = 365
matches['away_rest_days'] = 365
matches['rest_diff']      = 0

for idx, row in matches.iterrows():
    home, away = row['home_team'], row['away_team']
    dt = row['Date']
    h_rest = min((dt - last_match_date[home]).days, 365) if home in last_match_date else 365
    a_rest = min((dt - last_match_date[away]).days, 365) if away in last_match_date else 365
    matches.loc[idx, 'home_rest_days'] = h_rest
    matches.loc[idx, 'away_rest_days'] = a_rest
    matches.loc[idx, 'rest_diff']      = h_rest - a_rest
    last_match_date[home] = dt
    last_match_date[away] = dt

# Confederation strength on matches
matches['home_conf_score'] = matches['home_team'].map(conf_score)
matches['away_conf_score'] = matches['away_team'].map(conf_score)
matches['conf_diff']       = matches['home_conf_score'] - matches['away_conf_score']

# FIFA rank columns on matches
matches['home_fifa_rank']  = matches['home_team'].map(fifa_rank_dict).fillna(150)
matches['away_fifa_rank']  = matches['away_team'].map(fifa_rank_dict).fillna(150)
matches['home_fifa_score'] = matches['home_team'].map(fifa_score_dict).fillna(0.0)
matches['away_fifa_score'] = matches['away_team'].map(fifa_score_dict).fillna(0.0)

# Enrich team_dict with context features
for team in list(team_dict.keys()):
    team_dict[team]['conf_score'] = conf_score(team)
    team_dict[team]['rest_days']  = 7  # default for 2026 simulation

print('Contextual features added.')

Contextual features added.


In [77]:
default_team = {
    'elo': 1500, 'win_rate': 0.33, 'draw_rate': 0.33, 'loss_rate': 0.34,
    'goals_for': 1.0, 'goals_against': 1.0,
    'xg_for': 1.0, 'xg_against': 1.0, 'red_cards': 0.1,
    'fifa_rank': 150, 'fifa_pts': 1000, 'fifa_score': 0.0,
    'conf_score': 2, 'rest_days': 7,
}

features = [
    'home_elo', 'away_elo', 'elo_diff',
    'home_last5_win_rate', 'away_last5_win_rate',
    'home_last5_draw_rate', 'away_last5_draw_rate',
    'home_last5_loss_rate', 'away_last5_loss_rate',
    'home_last5_goals_for', 'away_last5_goals_for',
    'home_last5_goals_against', 'away_last5_goals_against',
    'form_diff', 'attack_diff', 'defense_diff', 'draw_rate_diff',
    # Step 1: xG + red cards
    'home_xg_for', 'away_xg_for',
    'home_xg_against', 'away_xg_against',
    'xg_diff',
    'home_red_cards_avg', 'away_red_cards_avg',
    # Step 4: FIFA ranking
    'home_fifa_rank', 'away_fifa_rank',
    'home_fifa_score', 'away_fifa_score',
    # Step 8: stage type
    'is_knockout',
    # Step 9: context
    'home_rest_days', 'away_rest_days', 'rest_diff',
    'home_conf_score', 'away_conf_score', 'conf_diff',
]

def build_match_features(home_team, away_team, is_knockout=0):
    h = team_dict.get(home_team, default_team)
    a = team_dict.get(away_team, default_team)
    return pd.DataFrame([{
        'home_elo':  h['elo'],  'away_elo':  a['elo'],
        'elo_diff':  h['elo'] - a['elo'],
        'home_last5_win_rate':      h['win_rate'],      'away_last5_win_rate':      a['win_rate'],
        'home_last5_draw_rate':     h['draw_rate'],     'away_last5_draw_rate':     a['draw_rate'],
        'home_last5_loss_rate':     h['loss_rate'],     'away_last5_loss_rate':     a['loss_rate'],
        'home_last5_goals_for':     h['goals_for'],     'away_last5_goals_for':     a['goals_for'],
        'home_last5_goals_against': h['goals_against'], 'away_last5_goals_against': a['goals_against'],
        'form_diff':      h['win_rate']      - a['win_rate'],
        'attack_diff':    h['goals_for']     - a['goals_for'],
        'defense_diff':   a['goals_against'] - h['goals_against'],
        'draw_rate_diff': h['draw_rate']     - a['draw_rate'],
        'home_xg_for':    h.get('xg_for',    h['goals_for']),
        'away_xg_for':    a.get('xg_for',    a['goals_for']),
        'home_xg_against':h.get('xg_against',h['goals_against']),
        'away_xg_against':a.get('xg_against',a['goals_against']),
        'xg_diff':         h.get('xg_for', h['goals_for']) - a.get('xg_for', a['goals_for']),
        'home_red_cards_avg': h.get('red_cards', 0),
        'away_red_cards_avg': a.get('red_cards', 0),
        'home_fifa_rank':   h.get('fifa_rank', 150),  'away_fifa_rank':   a.get('fifa_rank', 150),
        'home_fifa_score':  h.get('fifa_score', 0.0), 'away_fifa_score':  a.get('fifa_score', 0.0),
        'is_knockout':      is_knockout,
        'home_rest_days':   h.get('rest_days', 7),    'away_rest_days':   a.get('rest_days', 7),
        'rest_diff':        h.get('rest_days', 7)   - a.get('rest_days', 7),
        'home_conf_score':  h.get('conf_score', 2),   'away_conf_score':  a.get('conf_score', 2),
        'conf_diff':        h.get('conf_score', 2)  - a.get('conf_score', 2),
    }])

print(f'Total features: {len(features)}')

Total features: 35


In [78]:
def get_result(row):
    if row['home_score'] > row['away_score']: return 'Home'
    elif row['home_score'] < row['away_score']: return 'Away'
    return 'Draw'

matches['result'] = matches.apply(get_result, axis=1)

X_all = matches[features]
y_all = matches['result']

scaler = StandardScaler()
tscv   = TimeSeriesSplit(n_splits=5)

print('Time-series cross-validation:')
cv_scores = []
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_all)):
    X_tr, X_val = X_all.iloc[tr_idx], X_all.iloc[val_idx]
    y_tr, y_val = y_all.iloc[tr_idx], y_all.iloc[val_idx]
    fs = StandardScaler()
    fm = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)
    fm.fit(fs.fit_transform(X_tr), y_tr)
    acc = accuracy_score(y_val, fm.predict(fs.transform(X_val)))
    cv_scores.append(acc)
    print(f'  Fold {fold+1}: {acc:.3f}')

print(f'Mean CV accuracy: {np.mean(cv_scores):.3f} ± {np.std(cv_scores):.3f}')

# Final train/test split (temporal)
split   = int(len(matches) * 0.8)
train   = matches.iloc[:split]
test    = matches.iloc[split:]

X_train_s = scaler.fit_transform(train[features])
X_test_s  = scaler.transform(test[features])
y_train   = train['result']
y_test    = test['result']

base_lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)
base_lr.fit(X_train_s, y_train)
print('\nBase LR test-set performance:')
print(classification_report(y_test, base_lr.predict(X_test_s)))

Time-series cross-validation:
  Fold 1: 0.294
  Fold 2: 0.338
  Fold 3: 0.419
  Fold 4: 0.381
  Fold 5: 0.394
Mean CV accuracy: 0.365 ± 0.044

Base LR test-set performance:
              precision    recall  f1-score   support

        Away       0.44      0.62      0.52        69
        Draw       0.23      0.34      0.27        41
        Home       0.53      0.22      0.31        83

    accuracy                           0.39       193
   macro avg       0.40      0.39      0.37       193
weighted avg       0.43      0.39      0.38       193



In [79]:
calibrated_lr = CalibratedClassifierCV(
    LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42),
    method='isotonic', cv=5
)
calibrated_lr.fit(X_train_s, y_train)

raw_probs = base_lr.predict_proba(X_test_s)
cal_probs = calibrated_lr.predict_proba(X_test_s)
print('Raw  probs mean per class:', raw_probs.mean(axis=0).round(3))
print('Cal  probs mean per class:', cal_probs.mean(axis=0).round(3))
print('\nCalibrated LR test-set performance:')
print(classification_report(y_test, calibrated_lr.predict(X_test_s)))

Raw  probs mean per class: [0.488 0.296 0.216]
Cal  probs mean per class: [0.391 0.252 0.356]

Calibrated LR test-set performance:
              precision    recall  f1-score   support

        Away       0.40      0.55      0.47        69
        Draw       0.00      0.00      0.00        41
        Home       0.49      0.55      0.52        83

    accuracy                           0.44       193
   macro avg       0.30      0.37      0.33       193
weighted avg       0.36      0.44      0.39       193



In [80]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

model_rf = RandomForestClassifier(
    n_estimators=500, max_depth=10, min_samples_leaf=5,
    class_weight='balanced_subsample', random_state=42
)
model_rf.fit(train[features], y_train)

# ensemble_models list: (name, model, use_scaled_input)
ensemble_models = [
    ('lr', calibrated_lr, True),
    ('rf', model_rf, False),
]

try:
    from xgboost import XGBClassifier
    model_xgb = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss', random_state=42, verbosity=0
    )
    model_xgb.fit(X_train_s, y_train_enc)
    ensemble_models.append(('xgb', model_xgb, True))
    print('XGBoost added to ensemble.')
except ImportError:
    print('XGBoost not installed — skipping (pip install xgboost)')

try:
    from lightgbm import LGBMClassifier
    model_lgb = LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=42, verbose=-1
    )
    model_lgb.fit(X_train_s, y_train_enc)
    ensemble_models.append(('lgb', model_lgb, True))
    print('LightGBM added to ensemble.')
except ImportError:
    print('LightGBM not installed — skipping (pip install lightgbm)')

lr_classes  = list(calibrated_lr.classes_)
enc_classes = list(le.classes_)

def ensemble_predict_proba(X_raw, X_scaled):
    # Always wrap in named DataFrame so all models receive feature names
    X_df    = pd.DataFrame(X_raw,    columns=features)
    X_df_sc = pd.DataFrame(X_scaled, columns=features)
    probs = []
    for name, m, use_scaled in ensemble_models:
        X_in = X_df_sc if use_scaled else X_df
        p    = m.predict_proba(X_in)
        if name in ('xgb', 'lgb'):
            reorder = [enc_classes.index(c) for c in lr_classes]
            p = p[:, reorder]
        elif name == 'rf':
            rf_classes = list(model_rf.classes_)
            reorder    = [rf_classes.index(c) for c in lr_classes]
            p = p[:, reorder]
        probs.append(p)
    return np.mean(probs, axis=0)

def ensemble_predict(X_raw, X_scaled):
    proba = ensemble_predict_proba(X_raw, X_scaled)
    return np.array(lr_classes)[np.argmax(proba, axis=1)]

y_pred_ens = ensemble_predict(test[features].values, X_test_s)
print('\n=== ENSEMBLE TEST-SET PERFORMANCE ===')
print(classification_report(y_test, y_pred_ens))

# Feature importance from RF
importance = pd.DataFrame({'Feature': features, 'Importance': model_rf.feature_importances_})
print(importance.sort_values('Importance', ascending=False).head(10).to_string(index=False))

XGBoost added to ensemble.
LightGBM added to ensemble.

=== ENSEMBLE TEST-SET PERFORMANCE ===
              precision    recall  f1-score   support

        Away       0.45      0.68      0.54        69
        Draw       0.08      0.05      0.06        41
        Home       0.52      0.40      0.45        83

    accuracy                           0.42       193
   macro avg       0.35      0.38      0.35       193
weighted avg       0.40      0.42      0.40       193

             Feature  Importance
            elo_diff    0.073761
            away_elo    0.058867
            home_elo    0.053353
         home_xg_for    0.045613
home_last5_goals_for    0.044967
             xg_diff    0.038439
         attack_diff    0.037696
           form_diff    0.037031
home_last5_loss_rate    0.034892
      draw_rate_diff    0.032992


In [81]:
league_avg_goals = matches['home_score'].mean()

attack_rating  = {}
defense_rating = {}

for team, history in team_history.items():
    if len(history) == 0:
        attack_rating[team]  = league_avg_goals
        defense_rating[team] = league_avg_goals
        continue
    recent  = history[-10:]
    n       = len(recent)
    weights = np.array([0.9 ** (n - 1 - i) for i in range(n)])
    weights /= weights.sum()
    attack_rating[team]  = float(np.dot(weights, [h['goals_for']     for h in recent]))
    defense_rating[team] = float(np.dot(weights, [h['goals_against'] for h in recent]))

def poisson_xg(home, away):
    """Dixon-Coles-style expected goals with 10% home advantage."""
    ha = attack_rating.get(home,  league_avg_goals)
    hd = defense_rating.get(home, league_avg_goals)
    aa = attack_rating.get(away,  league_avg_goals)
    ad = defense_rating.get(away, league_avg_goals)
    home_xg = max(0.3, (ha * ad / league_avg_goals) * 1.1)
    away_xg = max(0.3,  aa * hd / league_avg_goals)
    return home_xg, away_xg

print(f'League avg goals/game: {league_avg_goals:.2f}')
for t in ['Brazil', 'France', 'Spain', 'Argentina', 'England']:
    print(f'  {t:12s}  attack={attack_rating.get(t, 0):.2f}  defense={defense_rating.get(t, 0):.2f}')

League avg goals/game: 1.78
  Brazil        attack=1.61  defense=0.66
  France        attack=2.31  defense=1.12
  Spain         attack=1.83  defense=1.02
  Argentina     attack=2.16  defense=1.52
  England       attack=1.84  defense=0.98


In [82]:
def simulate_match(home, away, is_knockout=0):
    """Simulate one match using Poisson draws from attack/defense ratings."""
    home_xg, away_xg = poisson_xg(home, away)
    home_goals = int(np.random.poisson(home_xg))
    away_goals = int(np.random.poisson(away_xg))
    return home, away, home_goals, away_goals

def simulate_knockout_match(home, away):
    """Knockout match — extra time then Elo-tilted penalties on a draw."""
    _, _, hg, ag = simulate_match(home, away, is_knockout=1)
    if hg > ag: return home
    if ag > hg: return away
    # Extra time
    et_h = int(np.random.poisson(0.3))
    et_a = int(np.random.poisson(0.3))
    if et_h > et_a: return home
    if et_a > et_h: return away
    # Penalties
    h_elo  = team_dict.get(home, default_team)['elo']
    a_elo  = team_dict.get(away, default_team)['elo']
    p_home = float(np.clip(0.5 + (h_elo - a_elo) * 0.00005, 0.3, 0.7))
    return np.random.choice([home, away], p=[p_home, 1 - p_home])

def update_points(table, home, away, home_goals, away_goals):
    if home_goals > away_goals:   table[home]['pts'] += 3
    elif home_goals < away_goals: table[away]['pts'] += 3
    else:
        table[home]['pts'] += 1
        table[away]['pts'] += 1
    table[home]['gf'] += home_goals;  table[home]['gd'] += home_goals - away_goals
    table[away]['gf'] += away_goals;  table[away]['gd'] += away_goals - home_goals

def simulate_group(fixtures):
    teams = pd.concat([fixtures['home_team'], fixtures['away_team']]).unique()
    table = {t: {'pts': 0, 'gd': 0, 'gf': 0} for t in teams}
    for _, row in fixtures.iterrows():
        _, _, hg, ag = simulate_match(row['home_team'], row['away_team'], is_knockout=0)
        update_points(table, row['home_team'], row['away_team'], hg, ag)
    return table

def rank_group(table):
    return sorted(table.items(), key=lambda x: (x[1]['pts'], x[1]['gd'], x[1]['gf']), reverse=True)

def simulate_knockout(teams):
    round_teams = list(teams)
    while len(round_teams) > 1:
        next_round = []
        for i in range(0, len(round_teams), 2):
            next_round.append(simulate_knockout_match(round_teams[i], round_teams[i+1]))
        round_teams = next_round
    return round_teams[0]

print('Simulation functions ready.')

Simulation functions ready.


In [83]:
# ── Group stage setup ─────────────────────────────────────────────────────────

groups_tables = {
    'A': {'Mexico': {}, 'South Africa': {}, 'Korea Republic': {}, 'Czech Republic': {}},
    'B': {'Canada': {}, 'Bosnia and Herzegovina': {}, 'Qatar': {}, 'Switzerland': {}},
    'C': {'Brazil': {}, 'Morocco': {}, 'Haiti': {}, 'Scotland': {}},
    'D': {'USA': {}, 'Paraguay': {}, 'Australia': {}, 'Türkiye': {}},
    'E': {'Germany': {}, 'Curacao': {}, "Côte d'Ivoire": {}, 'Ecuador': {}},
    'F': {'Netherlands': {}, 'Japan': {}, 'Sweden': {}, 'Tunisia': {}},
    'G': {'Belgium': {}, 'Egypt': {}, 'IR Iran': {}, 'New Zealand': {}},
    'H': {'Spain': {}, 'Cape Verde': {}, 'Saudi Arabia': {}, 'Uruguay': {}},
    'I': {'France': {}, 'Senegal': {}, 'Iraq': {}, 'Norway': {}},
    'J': {'Argentina': {}, 'Algeria': {}, 'Austria': {}, 'Jordan': {}},
    'K': {'Portugal': {}, 'Congo': {}, 'Uzbekistan': {}, 'Colombia': {}},
    'L': {'England': {}, 'Croatia': {}, 'Ghana': {}, 'Panama': {}},
}

group_map = {clean_team(t): g for g, teams in groups_tables.items() for t in teams}
schedule_2026['group'] = schedule_2026['home_team'].map(group_map)

group_fixtures = {
    g: schedule_2026[schedule_2026['group'] == g][['home_team', 'away_team']].reset_index(drop=True)
    for g in groups_tables
}

print('Groups configured. Sample Group C fixtures:')
print(group_fixtures['C'])

Groups configured. Sample Group C fixtures:
  home_team away_team
0    Brazil   Morocco
1     Haiti  Scotland
2  Scotland   Morocco
3    Brazil     Haiti
4  Scotland    Brazil
5   Morocco     Haiti


In [84]:
# ── Monte Carlo World Cup simulation ─────────────────────────────────────────
# WC 2026 format: 12 groups → top 2 advance (24) + best 8 third-place = 32

def simulate_world_cup():
    top2_teams  = []   # guaranteed qualifiers
    third_place = []   # (pts, gd, gf, team) for best-3rd selection

    for g in groups_tables:
        ranking = rank_group(simulate_group(group_fixtures[g]))
        top2_teams.extend([ranking[0][0], ranking[1][0]])
        # collect third-place finisher with their stats
        t3_team, t3_stats = ranking[2]
        third_place.append((t3_stats['pts'], t3_stats['gd'], t3_stats['gf'], t3_team))

    # Best 8 third-place teams advance (24 + 8 = 32)
    third_place.sort(reverse=True)
    best8_third = [entry[3] for entry in third_place[:8]]

    knockout_teams = top2_teams + best8_third
    return simulate_knockout(knockout_teams)

def run_monte_carlo(n_simulations=2000):
    results = Counter()
    for _ in range(n_simulations):
        results[simulate_world_cup()] += 1
    return {t: c / n_simulations for t, c in results.most_common()}

print('Running 2000 simulations...')
mc_results = run_monte_carlo(2000)

print('\n=== World Cup 2026 Win Probabilities ===')
for team, prob in sorted(mc_results.items(), key=lambda x: -x[1]):
    bar = '\u2588' * int(prob * 50)
    print(f'{team:30s} {prob*100:5.1f}%  {bar}')

Running 2000 simulations...

=== World Cup 2026 Win Probabilities ===
Netherlands                     15.2%  ███████
Brazil                          13.8%  ██████
France                          12.2%  ██████
Colombia                         7.0%  ███
England                          6.2%  ███
Belgium                          5.9%  ██
Spain                            5.5%  ██
Argentina                        4.2%  ██
Portugal                         3.4%  █
Germany                          2.9%  █
Uruguay                          2.5%  █
Bosnia and Herzegovina           2.4%  █
Türkiye                          2.4%  █
Paraguay                         1.8%  
Ecuador                          1.8%  
Norway                           1.2%  
Croatia                          1.1%  
Cape Verde                       1.1%  
Côte d'Ivoire                    0.9%  
Jordan                           0.8%  
Morocco                          0.8%  
Sweden                           0.7%  
Ghana         

In [85]:
# ── Group-stage win probability table (ensemble) ─────────────────────────────

pred_rows = []
for _, row in schedule_2026.iterrows():
    X_raw = build_match_features(row['home_team'], row['away_team'], is_knockout=0)
    X_sc  = scaler.transform(X_raw)
    proba = ensemble_predict_proba(X_raw.values, X_sc)[0]
    prob_dict = dict(zip(lr_classes, proba))
    pred_rows.append({
        'home_team':  row['home_team'],
        'away_team':  row['away_team'],
        'Home Win %': round(prob_dict.get('Home', 0) * 100, 1),
        'Draw %':     round(prob_dict.get('Draw', 0) * 100, 1),
        'Away Win %': round(prob_dict.get('Away', 0) * 100, 1),
    })

predictions = pd.DataFrame(pred_rows)
print(predictions.to_string(index=False))

             home_team              away_team  Home Win %  Draw %  Away Win %
                Mexico           South Africa        44.1    25.3        30.6
        Korea Republic         Czech Republic        33.4    32.0        34.5
                Canada Bosnia and Herzegovina        39.4    28.9        31.7
                   USA               Paraguay        44.2    21.4        34.4
                 Qatar            Switzerland        25.2    22.9        51.8
                Brazil                Morocco        37.9    22.6        39.6
                 Haiti               Scotland        14.6    38.8        46.5
             Australia                Türkiye        23.3    32.8        43.9
               Germany                Curacao        61.7    22.8        15.4
           Netherlands                  Japan        33.8    26.6        39.6
         Côte d'Ivoire                Ecuador        22.5    26.6        50.9
                Sweden                Tunisia        34.3    43.